# Extração SQL Server → MinIO (landing-zone)

Descobre tabelas via `INFORMATION_SCHEMA`, exporta cada uma para CSV em `s3a://landing-zone/<schema>_<tabela>/` usando PySpark + S3A (MinIO).

In [1]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv

# Carrega .env a partir da raiz do projeto (pai da pasta notebook/)
_root = Path.cwd()
if _root.name == "notebook":
    _root = _root.parent
load_dotenv(_root / ".env")

required_minio = ["MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD", "MINIO_S3_ENDPOINT"]
missing = [k for k in required_minio if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Defina no .env: {', '.join(missing)}")

jdbc_url_env = os.getenv("MSSQL_JDBC_URL", "").strip()
if jdbc_url_env:
    JDBC_URL = jdbc_url_env
else:
    host = os.getenv("MSSQL_JDBC_HOST", "localhost")
    port = os.getenv("MSSQL_JDBC_PORT", "1433")
    database = os.getenv("MSSQL_JDBC_DATABASE", "master")
    enc = os.getenv("MSSQL_JDBC_ENCRYPT", "false")
    trust = os.getenv("MSSQL_JDBC_TRUST_SERVER_CERTIFICATE", "true")
    JDBC_URL = (
        f"jdbc:sqlserver://{host}:{port};databaseName={database};"
        f"encrypt={enc};trustServerCertificate={trust}"
    )

JDBC_USER = os.getenv("MSSQL_JDBC_USER", "sa")
JDBC_PASSWORD = os.getenv("MSSQL_JDBC_PASSWORD") or os.getenv("MSSQL_SA_PASSWORD")
if not JDBC_PASSWORD:
    raise RuntimeError("Defina MSSQL_JDBC_PASSWORD ou MSSQL_SA_PASSWORD no .env")

MINIO_ENDPOINT = os.getenv("MINIO_S3_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")

LANDING_BUCKET = "landing-zone"

In [2]:
from pyspark.sql import SparkSession

# Pacotes compatíveis com Spark 3.5.x (Hadoop 3.3.4 interno)
packages = (
    "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11,"
    "io.delta:delta-spark_2.12:3.2.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
)

spark = (
    SparkSession.builder.appName("extracao-sqlserver-landing-zone")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.jars.packages", packages)
    # MinIO via S3A
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark inicializado:", spark.version)

your 131072x1 screen size is bogus. expect trouble
26/05/03 00:02:25 WARN Utils: Your hostname, DESKTOP-G0CPIMV resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/03 00:02:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ana/DEV/trabalho2ed-minio-sql/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ana/.ivy2/cache
The jars for the packages stored in: /home/ana/.ivy2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-07ad6c9e-cf36-4d44-935e-d392b5e5fdfb;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 1034ms :: artifacts dl 23ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.microsoft.

Spark inicializado: 3.5.3


In [3]:
jdbc_props = {
    "user": JDBC_USER,
    "password": JDBC_PASSWORD,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

tables_subquery = (
    "(SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
    "WHERE TABLE_TYPE = 'BASE TABLE' "
    "AND TABLE_SCHEMA NOT IN ('sys', 'INFORMATION_SCHEMA')) AS user_tables"
)

tables_df = spark.read.jdbc(url=JDBC_URL, table=tables_subquery, properties=jdbc_props)
table_rows = tables_df.collect()
print(f"Tabelas encontradas: {len(table_rows)}")
for r in table_rows:
    print(f"  - [{r.TABLE_SCHEMA}].[{r.TABLE_NAME}]")

Tabelas encontradas: 5
  - [dbo].[spt_fallback_db]
  - [dbo].[spt_fallback_dev]
  - [dbo].[spt_fallback_usg]
  - [dbo].[MSreplication_options]
  - [dbo].[spt_monitor]


In [5]:
# Verificar o que existe no SQL Server
query_bancos = "(SELECT name FROM sys.databases) AS bancos"

bancos = spark.read.jdbc(url=JDBC_URL, table=query_bancos, properties=jdbc_props).collect()

print("Bancos existentes:")
for b in bancos:
    print(f"  → {b.name}")

Bancos existentes:
  → master
  → tempdb
  → model
  → msdb
  → BibliotecaDb


In [6]:
# Verificar tabelas dentro do BibliotecaDb
query_tabelas = """
    (
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM BibliotecaDb.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    ) AS tabelas
"""

tabelas = spark.read.jdbc(url=JDBC_URL, table=query_tabelas, properties=jdbc_props).collect()

print(f"Tabelas encontradas no BibliotecaDb: {len(tabelas)}")
for t in tabelas:
    print(f"  → [{t.TABLE_SCHEMA}].[{t.TABLE_NAME}]")

Tabelas encontradas no BibliotecaDb: 6
  → [dbo].[Categoria]
  → [dbo].[Autor]
  → [dbo].[Livro]
  → [dbo].[Membro]
  → [dbo].[Emprestimo]
  → [dbo].[Multa]


In [7]:
def sanitize_folder_name(schema: str, table: str) -> str:
    raw = f"{schema}_{table}"
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", raw)
    return safe.strip("_") or "table"


# Query corrigida apontando direto para BibliotecaDb
query_tabelas = """
    (
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM BibliotecaDb.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    ) AS tabelas
"""

table_rows = spark.read.jdbc(url=JDBC_URL, table=query_tabelas, properties=jdbc_props).collect()

success = []
errors = []
total_rows = 0

for row in table_rows:
    schema, table = row.TABLE_SCHEMA, row.TABLE_NAME
    fq = f"[BibliotecaDb].[{schema}].[{table}]"
    folder = sanitize_folder_name(schema, table)
    target = f"s3a://{LANDING_BUCKET}/{folder}/"
    print(f"Extraindo {fq} → {target}")
    try:
        df = spark.read.jdbc(url=JDBC_URL, table=fq, properties=jdbc_props)
        n = df.count()
        total_rows += n
        (
            df.write.mode("overwrite")
            .option("header", True)
            .csv(target)
        )
        success.append({"schema": schema, "table": table, "rows": n, "path": target})
        print(f"  OK — {n} linhas gravadas")
    except Exception as exc:
        errors.append({"schema": schema, "table": table, "error": str(exc)})
        print(f"  ERRO — {exc}")

print(f"\n=== Resumo ===")
print(f"Sucesso: {len(success)}")
print(f"Com erro: {len(errors)}")
print(f"Total de linhas extraídas: {total_rows}")

Extraindo [BibliotecaDb].[dbo].[Categoria] → s3a://landing-zone/dbo_Categoria/


26/05/03 00:12:04 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/03 00:12:08 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/03 00:12:10 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 5 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Autor] → s3a://landing-zone/dbo_Autor/


26/05/03 00:12:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/03 00:12:13 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 10 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Livro] → s3a://landing-zone/dbo_Livro/


...

  OK — 15 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Emprestimo] → s3a://landing-zone/dbo_Emprestimo/


26/05/03 00:12:37 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/03 00:12:37 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 30 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Multa] → s3a://landing-zone/dbo_Multa/


26/05/03 00:12:38 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/03 00:12:39 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 8 linhas gravadas

=== Resumo ===
Sucesso: 6
Com erro: 0
Total de linhas extraídas: 88


In [8]:
print("\n=== Resumo ===")
print("Sucesso:", len(success))
print("Com erro:", len(errors))
print("Total de linhas extraídas:", total_rows)
if errors:
    print("\nFalhas:")
    for e in errors:
        print(f"  [{e['schema']}].[{e['table']}]: {e['error']}")


=== Resumo ===
Sucesso: 6
Com erro: 0
Total de linhas extraídas: 88


In [9]:
def list_s3a_prefix(spark_session, uri: str, indent: str = "") -> None:
    """Lista caminhos sob um prefixo s3a usando a API Hadoop já configurada no Spark."""
    jvm = spark_session.sparkContext._jvm
    hadoop_conf = spark_session.sparkContext._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(hadoop_conf)
    if not fs.exists(path):
        print(f"{indent}(prefixo inexistente ou vazio)")
        return
    statuses = fs.listStatus(path)
    for st in statuses:
        p = st.getPath().toString()
        if st.isDirectory():
            print(f"{indent}{p}/")
            list_s3a_prefix(spark_session, p + "/", indent + "  ")
        else:
            print(f"{indent}{p}")


print(f"Objetos em s3a://{LANDING_BUCKET}/")
list_s3a_prefix(spark, f"s3a://{LANDING_BUCKET}/")

Objetos em s3a://landing-zone/
s3a://landing-zone/dbo_Autor/
  s3a://landing-zone/dbo_Autor/_SUCCESS
  s3a://landing-zone/dbo_Autor/part-00000-a563abd9-34cf-4eff-8c1a-db492e080cf4-c000.csv
s3a://landing-zone/dbo_Categoria/
  s3a://landing-zone/dbo_Categoria/_SUCCESS
  s3a://landing-zone/dbo_Categoria/part-00000-d434963b-4ac4-4272-89e7-94926c503be5-c000.csv
s3a://landing-zone/dbo_Emprestimo/
  s3a://landing-zone/dbo_Emprestimo/_SUCCESS
  s3a://landing-zone/dbo_Emprestimo/part-00000-f8de8411-913f-4b71-9d72-7c7e53de3e02-c000.csv
s3a://landing-zone/dbo_Livro/
  s3a://landing-zone/dbo_Livro/_SUCCESS
  s3a://landing-zone/dbo_Livro/part-00000-e568959d-a315-4952-ac1b-37b5593e76e9-c000.csv
s3a://landing-zone/dbo_MSreplication_options/
  s3a://landing-zone/dbo_MSreplication_options/_SUCCESS
  s3a://landing-zone/dbo_MSreplication_options/part-00000-18ba9ee9-3c29-4fe7-bbf0-666c98376397-c000.csv
s3a://landing-zone/dbo_Membro/
  s3a://landing-zone/dbo_Membro/_SUCCESS
  s3a://landing-zone/dbo_Membro/

In [10]:
spark.stop()